# Lesson 03 - Agentic Design Patterns

ဤသင်ခန်းစာတွင် ကျွန်ုပ်တို့သည် ထိရောက်သော AI အေးဂျင်များ တည်ဆောက်ရန် အခြေခံဒီဇိုင်းပုံစံသုံးမျိုးကို လေ့လာပါမည်-

1. **ရှင်းလင်းသည့် အေးဂျင်ညွှန်ကြားချက်များ** — အေးဂျင်၏ အပြုအမူကို လမ်းညွှန်ပေးရန် တိကျသော၊ တာဝန်သတ်မှတ်ထားသော prompt များကို ဖန်တီးခြင်း
2. **Pydantic မော်ဒယ်များဖြင့် ဖွဲ့စည်းထားသော ထွက်ရှိမှု** — အေးဂျင်များမှ ကြိုတင်ခန့်မှန်းနိုင်ပြီး သက်ဆိုင်သေချာစွာ စစ်ဆေးထားသော ဒေတာများကို ပြန်လည်ထုတ်ပေးရန်
3. **တစ်လုပ်ငန်းတာဝန်တည်းသော အေးဂျင်များ** — တစ်ခုချင်းစီကောင်းစွာ ပြုလုပ်နိုင်သော အာရုံစိုက်ထားသောအေးဂျင် များကို ဒီဇိုင်းဆွဲခြင်း

ကျွန်ုပ်တို့သည် ဒီဒီဇိုင်းပုံစံအားလုံးကို **ခရီးသွားတည်နေရာအကြံပြုစနစ်** စနစ်မှာ အသုံးချ၍ မှတ်တမ်းတင်ထားသောနေရာများကို အကြံပြုခြင်း၊ ရရှိနိုင်မှုစစ်ဆေးခြင်း နှင့် လိုဂျစ်စတစ်များကို ကိုင်တွယ်ပေးခြင်း စသည်ဖြင့် အဆင့်ဆင့်စနစ်တဝိုက်တည်ဆောက်သွားမည် ဖြစ်ပါသည်။


## ပြင်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: ပေတိန် ၁ - အေးဂျင့် အညွှန်းများအား တိကျရှင်းလင်းစွာ ရေးသားခြင်း

အကျိုးသက်ရောက်မှု အများဆုံး ပေတိန်မှာလည်း အလွယ်တကူဆုံးဖြစ်သည် - သင့်အေးဂျင့်အတွက် ရှင်းလင်းပြီး အသေးစိတ်ညွှန်းချက်များကို ရေးသားခြင်း။

ကောင်းမွန်သော အညွှန်းများမှာ:
- **ဘယ်သူ** သည် အေးဂျင့်ဖြစ်သည် (ပုဂ္ဂိုလ်ရေးနှင့် အသံအတိုးအကျယ်)
- **ဘာကို** လုပ်သင့်သည် (အဆင့်လိုက်တာဝန်များ)
- **ဘယ်လို** လုပ်ဆောင်သင့်သည် (ကန့်သတ်ချက်များနှင့် ပုံစံ)

အောက်တွင်၊ ငွေခွဲခရီးသွားအကြံပေး အေးဂျင့်တစ်ယောက်ကို တိကျရှင်းလင်းသော အညွှန်းများနှင့် ဖန်တီးထားပြီး ၎င်းထုတ်ပေးသည့် အဖြေတိုင်းကို ဦးတည်သတ်မှတ်ထားပါသည်။


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

Free-form text သည် ဆက်သွယ်ကြောင်းအတွက် အသုံးဝင်သော်လည်း၊ အောက်တိုဘာစနစ်များတွင် ဖွဲ့စည်းထားသော အချက်အလက်များ လိုအပ်သည်။
**Pydantic models** နှင့် **tool function** တို့ကို တွဲဖက် အသုံးပြုခြင်းဖြင့် -

- အေးဂျင့်၏ output အတွက် တိတိကျကျ schema တစ်ခု သတ်မှတ်နိုင်သည်။
- တုံ့ပြန်ချက်များကို အလိုအလျောက် စစ်ဆေးနိုင်သည်။
- အေးဂျင့်ရလဒ်များကို လျှောက်လွှာ logic ထဲတွင် ယုံကြည်စိတ်ချစွာ မြှင့်တင်နိုင်သည်။

တစ်ခုတည်း tool တစ်ခုကိုလည်း မိတ်ဆက်ပေးပါသည်၊ ၎င်းသည် အေးဂျင့်၏ အကြံပေးချက်များကို အချက်အလက်အစစ်အမှန်အပေါ် အခြေခံစေတော့မှာဖြစ်သည်။


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

ရှုပ်ထွေးသောလုပ်ငန်းများကို တာဝန်တစ်ခုချင်းစီ သတ်မှတ်ထားသော အေးဂျင့်များစွာဖြင့် ခွဲခြားလုပ်ဆောင်ခြင်းအားဖြင့် အကျိုးရှိပါသည်-

- အရပ်များနှင့် အရရှိနိုင်မှုများကို သိရှိထားသော **ခရိုင်ကျွမ်းကျင်သူ**
- လေယာဉ်၊ ဟိုတယ်နှင့် ခရီးစဉ်များကို ကိုင်တွယ်သည့် **ရေကြောင်းစီမံသူ**

ဒါဟာ *ဝေဖန်ခြင်းခြားနားမှု* ဆိုတဲ့ ဆော့ဖ်ဝဲ အင်ဂျင်နီယာ လမ်းညွှန်ချက်နဲ့ အညီဖြစ်ပြီး — အေးဂျင့်တိုင်း အလွယ်တကူ စမ်းသပ်၊ ပြုပြင်ထိန်းသိမ်း နှင့် တိုးတက်လာစေလို့ရပါတယ်။


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## အနှစ်ချုပ်

ဤသင်ခန်းစာတွင် ကျွန်ုပ်တို့သည် ခရီးသွားအကြံပြုရေးကိရိယာအခြေအနေသို့ agentic ဒီဇိုင်းပုံစံသုံးမျိုးကို အသုံးပြုခဲ့သည်-

| ပုံစံ | အဓိကအယူအဆ | အကျိုးကျေးဇူး |
|---|---|---|
| **ရှင်းလင်းသောညွှန်ကြားချက်များ** | ပုဂ္ဂိုလ်ရည်၊ တာဝန်များနှင့် ကန့်သတ်ချက်များကို စတင်သတ်မှတ်ပါ | တစ်ကိုယ်ရည်၊ အမှတ်တံဆိပ်အညီ လှုပ်ရှားမှုခုခံ |
| **ဖွဲ့စည်းတည်ဆောက်ထားသော ထုတ်လွှင့်ချက်** | Pydantic မော်ဒယ်များကို တုံ့ပြန်ပုံစံအဖြစ် အသုံးပြုပါ | မှန်ကန်သတ်မှတ်ထားပြီး စက်ဖတ်နိုင်သော ရလဒ်များ |
| **တစ်ခုချင်းတာဝန်ခံမှု** | တစ်ဦးချင်းစီကို အာရုံစိုက်ထားသော အလုပ်တစ်ခုပေးပါ | စမ်းသပ်ရလွယ်ကူ၊ ပေါင်းစည်းစီမံရန်အဆင်ပြေ |

ဤပုံစံများသည် သဘာဝတရားအတိုင်း ပေါင်းစပ်နိုင်ပြီး — သင်သည်ရှင်းလင်းသောညွှန်ကြားချက်များကို ဖွဲ့စည်းတည်ဆောက်ထားသော ထုတ်လွှင့်ချက်နှင့် တစ်ခုချင်းတာဝန်ခံ agent အတွင်း ပေါင်းစပ်၍ ခိုင်ခံ့ပြီး ထုတ်လုပ်င်ရာစနစ်များ ဆောက်လုပ်နိုင်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
